# Support Vector Machine (SVM) Classification — Complete Guide

A comprehensive SVM deep-dive covering:
- **SVM Theory**: Hyperplane, Margin, Support Vectors
- **Kernel Trick**: Linear, RBF, Polynomial, Sigmoid
- **C & Gamma** hyperparameter tuning
- **Decision Boundary Visualization**
- **Multi-class Classification** with `SVC` and `LinearSVC`
- **Soft vs Hard Margin** comparison
- **Performance Comparison** across kernels

---

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.svm import SVC, LinearSVC
from sklearn.datasets import load_breast_cancer, make_moons, make_circles, make_classification
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (classification_report, confusion_matrix, 
                             accuracy_score, roc_curve, auc, RocCurveDisplay)
from sklearn.pipeline import Pipeline
import warnings
warnings.filterwarnings('ignore')

## 1. SVM Intuition — Finding the Optimal Hyperplane

SVM finds the hyperplane that **maximizes the margin** between two classes.

Key Terms:
- **Hyperplane**: Decision boundary (line in 2D, plane in 3D)
- **Support Vectors**: Data points closest to the hyperplane (they "support" it)
- **Margin**: Distance between the hyperplane and nearest support vectors
- **C (Regularization)**: Controls trade-off between margin width and misclassification
  - High C → Narrow margin, fewer errors (risk overfitting)
  - Low C → Wide margin, more errors allowed (better generalization)

In [ ]:
# Generate linearly separable data
np.random.seed(42)
X_linear, y_linear = make_classification(n_samples=100, n_features=2, n_redundant=0,
                                          n_informative=2, n_clusters_per_class=1,
                                          class_sep=2.0, random_state=42)

def plot_svm_decision_boundary(X, y, model, title, ax):
    """Plot SVM decision boundary with margin and support vectors."""
    h = 0.02
    x_min, x_max = X[:, 0].min() - 1, X[:, 0].max() + 1
    y_min, y_max = X[:, 1].min() - 1, X[:, 1].max() + 1
    xx, yy = np.meshgrid(np.arange(x_min, x_max, h), np.arange(y_min, y_max, h))
    Z = model.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    
    ax.contourf(xx, yy, Z, alpha=0.3, cmap='RdYlBu')
    ax.scatter(X[:, 0], X[:, 1], c=y, cmap='RdYlBu', edgecolors='black', s=50)
    # Highlight support vectors
    if hasattr(model, 'support_vectors_'):
        ax.scatter(model.support_vectors_[:, 0], model.support_vectors_[:, 1],
                   s=200, facecolors='none', edgecolors='black', linewidths=2,
                   label=f'Support Vectors ({len(model.support_vectors_)})')
    ax.set_title(title)
    ax.legend()

# Compare C values
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, C in zip(axes, [0.01, 1, 100]):
    svm = SVC(kernel='linear', C=C).fit(X_linear, y_linear)
    plot_svm_decision_boundary(X_linear, y_linear, svm, f'Linear SVM (C={C})', ax)
plt.suptitle('Effect of C Parameter on Decision Boundary', fontsize=14)
plt.tight_layout()
plt.show()

## 2. Kernel Trick — Handling Non-Linear Data

When data isn't linearly separable, SVM uses **kernels** to map data to a higher-dimensional space.

| Kernel | Formula | Best For |
|--------|---------|----------|
| Linear | $K(x,y) = x \cdot y$ | Linearly separable data |
| RBF (Gaussian) | $K(x,y) = e^{-\gamma\|x-y\|^2}$ | Most general-purpose (default) |
| Polynomial | $K(x,y) = (\gamma x \cdot y + r)^d$ | Known polynomial relationship |
| Sigmoid | $K(x,y) = \tanh(\gamma x \cdot y + r)$ | Neural network-like |

**Gamma** controls the influence of each training example:
- High gamma → Each point has close-range influence (complex boundary, risk overfitting)
- Low gamma → Each point has far-reaching influence (smoother boundary)

In [ ]:
# Non-linear datasets
X_moons, y_moons = make_moons(n_samples=300, noise=0.2, random_state=42)
X_circles, y_circles = make_circles(n_samples=300, noise=0.1, factor=0.5, random_state=42)

fig, axes = plt.subplots(2, 4, figsize=(20, 10))

for row, (X, y, name) in enumerate([(X_moons, y_moons, 'Moons'), (X_circles, y_circles, 'Circles')]):
    for col, kernel in enumerate(['linear', 'rbf', 'poly', 'sigmoid']):
        svm = SVC(kernel=kernel, gamma='scale').fit(X, y)
        acc = svm.score(X, y)
        plot_svm_decision_boundary(X, y, svm, f'{name} — {kernel.upper()} (acc={acc:.2f})', axes[row][col])

plt.suptitle('Kernel Comparison on Non-Linear Datasets', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Effect of Gamma on RBF kernel
fig, axes = plt.subplots(1, 4, figsize=(20, 5))
for ax, gamma in zip(axes, [0.01, 0.1, 1, 10]):
    svm = SVC(kernel='rbf', gamma=gamma, C=1).fit(X_moons, y_moons)
    acc = svm.score(X_moons, y_moons)
    plot_svm_decision_boundary(X_moons, y_moons, svm, f'gamma={gamma} (acc={acc:.2f})', ax)
plt.suptitle('Effect of Gamma on RBF Kernel', fontsize=14)
plt.tight_layout()
plt.show()

## 3. Real-World Classification — Breast Cancer Dataset

In [ ]:
# Load dataset
cancer = load_breast_cancer()
X, y = cancer.data, cancer.target
print(f'Features: {X.shape[1]} | Samples: {X.shape[0]}')
print(f'Classes: {cancer.target_names} | Distribution: {np.bincount(y)}')

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, 
                                                     random_state=42, stratify=y)
# Scaling is CRUCIAL for SVM
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
# Compare all kernels
results = {}
for kernel in ['linear', 'rbf', 'poly', 'sigmoid']:
    svm = SVC(kernel=kernel, gamma='scale', probability=True, random_state=42)
    svm.fit(X_train_scaled, y_train)
    
    train_acc = svm.score(X_train_scaled, y_train)
    test_acc = svm.score(X_test_scaled, y_test)
    cv_scores = cross_val_score(svm, X_train_scaled, y_train, cv=5, scoring='accuracy')
    
    if hasattr(svm, 'support_vectors_'):
        n_sv = len(svm.support_vectors_)
    else:
        n_sv = 'N/A'
    
    results[kernel] = {
        'Train Acc': train_acc,
        'Test Acc': test_acc,
        'CV Mean': cv_scores.mean(),
        'CV Std': cv_scores.std(),
        'Support Vectors': n_sv
    }
    print(f'{kernel.upper():8s} | Train: {train_acc:.4f} | Test: {test_acc:.4f} | '
          f'CV: {cv_scores.mean():.4f} ± {cv_scores.std():.4f} | SVs: {n_sv}')

results_df = pd.DataFrame(results).T
results_df

## 4. Hyperparameter Tuning with GridSearchCV

In [ ]:
# Grid Search for optimal C and gamma
param_grid = {
    'svm__C': [0.01, 0.1, 1, 10, 100],
    'svm__gamma': ['scale', 'auto', 0.001, 0.01, 0.1, 1],
    'svm__kernel': ['rbf', 'linear']
}

pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('svm', SVC(probability=True, random_state=42))
])

grid_search = GridSearchCV(pipeline, param_grid, cv=5, scoring='accuracy',
                           n_jobs=-1, verbose=0, return_train_score=True)
grid_search.fit(X_train, y_train)

print(f'Best Parameters: {grid_search.best_params_}')
print(f'Best CV Score: {grid_search.best_score_:.4f}')
print(f'Test Score: {grid_search.score(X_test, y_test):.4f}')

In [ ]:
# Heatmap of C vs Gamma performance (RBF kernel only)
rbf_results = pd.DataFrame(grid_search.cv_results_)
rbf_mask = rbf_results['param_svm__kernel'] == 'rbf'
rbf_only = rbf_results[rbf_mask].copy()

# Only plot if we have enough RBF results
if len(rbf_only) > 0:
    pivot = rbf_only.pivot_table(values='mean_test_score', 
                                 index='param_svm__C', 
                                 columns='param_svm__gamma')
    plt.figure(figsize=(10, 6))
    sns.heatmap(pivot, annot=True, fmt='.3f', cmap='YlGnBu', cbar_kws={'label': 'CV Accuracy'})
    plt.title('GridSearchCV: C vs Gamma (RBF Kernel)')
    plt.xlabel('Gamma')
    plt.ylabel('C')
    plt.tight_layout()
    plt.show()

In [ ]:
# Final evaluation with best model
best_svm = grid_search.best_estimator_
y_pred = best_svm.predict(X_test)

print('Classification Report (Best SVM):')
print(classification_report(y_test, y_pred, target_names=cancer.target_names))

# Confusion Matrix
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=cancer.target_names, yticklabels=cancer.target_names)
axes[0].set_title('Confusion Matrix')
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('Actual')

# ROC Curve
y_prob = best_svm.predict_proba(X_test)[:, 1]
fpr, tpr, _ = roc_curve(y_test, y_prob)
roc_auc = auc(fpr, tpr)
axes[1].plot(fpr, tpr, color='blue', linewidth=2, label=f'AUC = {roc_auc:.3f}')
axes[1].plot([0, 1], [0, 1], 'r--', alpha=0.5)
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('ROC Curve')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 5. Key Takeaways

| Aspect | Details |
|--------|--------|
| **When to Use SVM** | Medium-sized datasets, high-dimensional data, clear margin of separation |
| **Always Scale** | SVM is distance-based — `StandardScaler` is essential |
| **Kernel Selection** | Start with RBF (default); use Linear for high-dimensional/text data |
| **C Parameter** | Regularization: small C = wide margin (underfitting risk), large C = narrow margin (overfitting risk) |
| **Gamma** | Only for RBF/Poly/Sigmoid: small gamma = smooth boundary, large gamma = complex boundary |
| **Pros** | Effective in high dimensions, memory efficient (uses support vectors only) |
| **Cons** | Slow on large datasets (O(n²) to O(n³)), not great with noisy/overlapping classes |